# **Artificial Intelligence - Final Project**

**Integrated AI Approach for Telecommunication Customer Churn Analysis: A Comparison of Supervised Classification Method (ANN, Naive Bayes, Decision Tree)**

**The Member of the Group 5 of Final Project**

- Abdullah Al-Firdaus Nuzula 	(24031554008)
- Fio Ulaa' Octriyanti        (24031554030)
- M. Raffi Fahrezi           	(24031554100)

**Lecturer:**
- Riskyana Dewi Intan Puspitasari, M.Kom.
- Kartika Chandra Dewi, S.Si., M.Si.

# **Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, balanced_accuracy_score
)
from sklearn.pipeline import Pipeline

# **Dataset**

In [ ]:
splits = {'train': 'train.csv', 'validation': 'validation.csv', 'test': 'test.csv'}
df = pd.read_csv("hf://datasets/aai510-group1/telco-customer-churn/" + splits["train"])
df.to_csv("Churn_Dataset.csv", index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print(df)

      Age  Avg Monthly GB Download  Avg Monthly Long Distance Charges  \
0      72                        4                              19.44   
1      27                       59                              45.62   
2      59                        0                              16.07   
3      25                       27                               0.00   
4      31                       21                              17.22   
...   ...                      ...                                ...   
4220   36                        0                               7.76   
4221   77                       22                              23.43   
4222   56                        0                              28.06   
4223   45                       22                               0.00   
4224   45                       22                              12.69   

       Churn Category         Churn Reason  Churn Score              City  \
0                 NaN                  NaN    

In [ ]:
print("="*10, "Dataset Information", "="*10)
print("Dataset Columns:", df.shape[1])
print("Dataset Rows:", df.shape[0])

========== Dataset Information ==========
Dataset Columns: 52
Dataset Rows: 4225


## **Data Cleaning**

In [ ]:
nan_cols = df.isna().sum()
nan_cols[nan_cols > 0]

,0
Churn Category,3104
Churn Reason,3104
Internet Type,886
Offer,2324


In [ ]:
df.drop(["Churn Category",
         "Churn Reason",
         "Customer Status",
         "Customer ID", #we drop this because they are irrelevant for the model
         "Zip Code"],
        inplace=True,
        axis=1)
df.fillna({"Internet Type":"No Type Given"}, inplace=True)
df.fillna({"Offer":"No Offer Given"}, inplace=True)


In [ ]:
print("="*10, "Cleaned Dataset Information", "="*10)
print(f"Columns: {df.shape[1]}")
print(f"Rows: {df.shape[0]}")

========== Cleaned Dataset Information ==========
Columns: 47
Rows: 4225


## **Imbalance**

In [ ]:
threshold = 0.7

biased_cols = []

for col in df.columns:
  vc = df[col].value_counts(normalize=True)

  max_ratio = vc.max()

  if max_ratio > threshold:
    biased_cols.append(col)
    print(f"\n>>> Imbalanced Feature: {col}")
    print(vc)
    if max_ratio > 0.90:
      df.drop(col, inplace=True, axis=1)
      print(f"Imbalanced Feature, Dropped {col}!")
      biased_cols.remove(col)


>>> Imbalanced Feature: Country
Country
United States    1.0
Name: proportion, dtype: float64
Imbalanced Feature, Dropped Country!

>>> Imbalanced Feature: Dependents
Dependents
0    0.766864
1    0.233136
Name: proportion, dtype: float64

>>> Imbalanced Feature: Internet Service
Internet Service
1    0.790296
0    0.209704
Name: proportion, dtype: float64

>>> Imbalanced Feature: Number of Dependents
Number of Dependents
0    0.766864
1    0.079763
3    0.074793
2    0.074556
5    0.001893
4    0.001657
6    0.000237
8    0.000237
Name: proportion, dtype: float64

>>> Imbalanced Feature: Online Security
Online Security
0    0.707456
1    0.292544
Name: proportion, dtype: float64

>>> Imbalanced Feature: Phone Service
Phone Service
1    0.899408
0    0.100592
Name: proportion, dtype: float64

>>> Imbalanced Feature: Premium Tech Support
Premium Tech Support
0    0.703432
1    0.296568
Name: proportion, dtype: float64

>>> Imbalanced Feature: Quarter
Quarter
Q3    1.0
Name: proportion,

In [ ]:
print("="*10, "Biased Features", "="*10)
for i in biased_cols:
  print(i)

========== Biased Features ==========
Dependents
Internet Service
Number of Dependents
Online Security
Phone Service
Premium Tech Support
Senior Citizen
Total Extra Data Charges
Under 30
Churn


# **Models**

## **Feature Engineering**

In [ ]:
txt_cols = []

print("="*10, "Text Features", "="*10)
for col in df.columns:
  if df[col].dtype == "object":
    txt_cols.append(col)
    print(col)

========== Text Features ==========
City
Contract
Gender
Internet Type
Lat Long
Offer
Payment Method


In [ ]:
num_cols = []

print("="*10, "Numerical Features", "="*10)
for col in df.columns:
  if len(df[col].unique()) > 2 and df[col].dtype != "object":
    num_cols.append(col)
    print(col)

========== Numerical Features ==========
Age
Avg Monthly GB Download
Avg Monthly Long Distance Charges
Churn Score
CLTV
Latitude
Longitude
Monthly Charge
Number of Dependents
Number of Referrals
Population
Satisfaction Score
Tenure in Months
Total Charges
Total Extra Data Charges
Total Long Distance Charges
Total Revenue


In [ ]:
binary_cols = []

print("="*10, "Binary Features", "="*10)
for col in df.columns:
  if len(df[col].unique()) == 2 and df[col].dtype != "object":
    binary_cols.append(col)
    print(col)

========== Binary Features ==========
Dependents
Device Protection Plan
Internet Service
Married
Multiple Lines
Online Backup
Online Security
Paperless Billing
Partner
Phone Service
Premium Tech Support
Referred a Friend
Senior Citizen
Streaming Movies
Streaming Music
Streaming TV
Under 30
Unlimited Data
Churn


In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

X_train_processed = preprocess.fit_transform(X_train_raw)
X_test_processed = preprocess.transform(X_test_raw)

feature_names = (list(preprocess.named_transformers_['cat'].get_feature_names_out(cat_cols)) +
                 list(num_cols))

print("Jumlah Fitur Final:", len(feature_names))
print(feature_names)

NameError: name 'cat_cols' is not defined

## **Decision Tree**

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_encoded = preprocess.fit_transform(X_train_raw)
X_test_encoded = preprocess.transform(X_test_raw)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_encoded, y_train)

print("="*10, "SHAPE INFO", "="*10)
print("X_train_raw      :", X_train_raw.shape)
print("X_train_encoded  :", X_train_encoded.shape)
print("X_train_resampled:", X_train_sm.shape, "(Setelah SMOTE)")
print("-" * 30)
print("X_test_encoded   :", X_test_encoded.shape)
print("y_train_resampled:", y_train_sm.shape)
print("y_test           :", y_test.shape)

print("\n")
print("="*10, "TARGET DISTRIBUTION", "="*10)
print("Original Train Set (Sebelum SMOTE):\n", y_train.value_counts(normalize=True))
print("\nFinal Train Set (Setelah SMOTE):\n", y_train_sm.value_counts(normalize=True))
print("\nTest Set :\n", y_test.value_counts(normalize=True))

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_sm, y_train_sm)

y_pred = dt.predict(X_test_encoded)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Contoh untuk Decision Tree
print("Mencari parameter terbaik untuk Decision Tree...")
param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='recall')
grid_search.fit(X_train_sm, y_train_sm)

print("Parameter Terbaik:", grid_search.best_params_)
print("Skor Recall Terbaik:", grid_search.best_score_)

# Gunakan model terbaik ini untuk prediksi selanjutnya
dt_best = grid_search.best_estimator_

In [ ]:
print("="*10, "Balanced Accuracy", "="*10, "\n", balanced_accuracy_score(y_test, y_pred))

importances = dt.feature_importances_
indices = np.argsort(importances)[::-1]
print("\n")
print("="*10, "Top 20 Feature Importances:", "="*10)
for i in indices[:20]:
    print(i, importances[i])

### **Evaluation Metrics for Decision Tree**

In [ ]:
print("="*10, "EVALUATION METRICS", "="*10)
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'],
            annot_kws={"size": 14})

plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label (Actual)', fontsize=12)
plt.title('Confusion Matrix Heatmap', fontsize=14)
plt.show()

y_prob = dt.predict_proba(X_test_encoded)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## **Naive Bayes**

In [ ]:
nb = GaussianNB()

print ("Training Naive Bayes model... ")
nb.fit(X_train_sm, y_train_sm)

In [ ]:
y_pred_nb = nb.predict(X_test_encoded)

print("="*10, "NAIVE BAYES EVALUATION METRICS", "="*10)
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_nb):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))

### **Evaluation Metrics for Naive Bayes**

In [ ]:
print("\nConfusion Matrix:\n")
cm_nb = confusion_matrix(y_test, y_pred_nb)
print(cm_nb)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Naive Bayes')
plt.show()

## **Artificial Neural Network (MLP Classifier)**

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
ann_model = ImbPipeline(steps=[
    ('preprocess', preprocess),       # Scaling & Encoding
    ('smote', SMOTE(random_state=42)), # SMOTE otomatis hanya saat fit()
    ('model', MLPClassifier(hidden_layer_sizes=(64, 32),
                            activation='relu',
                            solver='adam',
                            max_iter=500,
                            random_state=42))
])

In [ ]:
print("Training ANN with Pipeline...")
ann_model.fit(X_train_raw, y_train)

In [ ]:
y_pred_ann = ann_model.predict(X_test_raw)

In [ ]:
print("="*10, "ANN PIPELINE RESULTS", "="*10)
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_ann):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_ann))

print("\nConfusion Matrix:\n")
cm_ann = confusion_matrix(y_test, y_pred_ann)
print(cm_ann)

### **Evaluation Metrics for Artificial Neural Network (MLP Classifier)**

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - ANN (Pipeline)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()